In [1]:
import os
import numpy as np
import pandas as pd
 
PATH = "american_bankruptcy.csv"
os.makedirs("results", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
 
NAMES = ["current_assets", "cogs", "depreciation_amortization", "ebitda",
         "inventory", "net_income", "total_receivables", "market_value",
         "net_sales", "total_assets", "long_term_debt", "ebit",
         "gross_profit", "current_liabilities", "retained_earnings",
         "total_revenue", "total_liabilities", "operating_expenses"]
RENAME = {f"X{i+1}": n for i, n in enumerate(NAMES)}
 
df = pd.read_csv(PATH).rename(columns=RENAME)
df = df.sort_values(["company_name", "year"]).reset_index(drop=True)
df["failed_co"] = (df["status_label"] == "failed").astype(int)
 
# ---------------------------------------------------------------- 1. label
print("=== 1. Is status_label constant within a company? ===")
n_lab = df.groupby("company_name")["status_label"].nunique()
print("Companies with BOTH labels:", int((n_lab > 1).sum()),
      "of", len(n_lab))
 
gaps = df.groupby("company_name")["year"].apply(
    lambda s: int((s.diff().dropna() != 1).sum()))
print("Companies with a gap in years:", int((gaps > 0).sum()))
 
# ------------------------------------------------------- 2. mapping checks
print("\n=== 2. Accounting-identity checks on the X->name mapping ===")
tol = 0.01  # tolerance relative to max(|value|, 1)
 
 
def frac_close(a, b):
    return float((np.abs(a - b) <= tol * np.maximum(np.abs(b), 1)).mean())
 
 
print("net_sales == total_revenue      :",
      frac_close(df.net_sales, df.total_revenue))
print("ebitda == ebit + D&A            :",
      frac_close(df.ebitda, df.ebit + df.depreciation_amortization))
print("gross_profit == net_sales - cogs:",
      frac_close(df.gross_profit, df.net_sales - df.cogs))
print("opex == total_revenue - ebitda  :",
      frac_close(df.operating_expenses, df.total_revenue - df.ebitda))
print("share CL + LTD <= total_liab    :",
      float((df.current_liabilities + df.long_term_debt
             <= df.total_liabilities * 1.01).mean()))
print("share inv + recv <= curr_assets :",
      float((df.inventory + df.total_receivables
             <= df.current_assets * 1.01).mean()))
print("share curr_assets <= total_assets:",
      float((df.current_assets <= df.total_assets * 1.01).mean()))
print("share total_liab <= 3*total_assets (loose):",
      float((df.total_liabilities <= 3 * df.total_assets).mean()))
 
# -------------------------------------------------------------- 3. targets
# Working assumption (VERIFY in section 4): the last observed row of a
# failed company is the fiscal year before the bankruptcy filing.
last_year = df.groupby("company_name")["year"].transform("max")
df["years_to_end"] = last_year - df["year"]          # 0 = last row
 
for h in (1, 2, 3):
    df[f"y{h}"] = ((df.failed_co == 1) & (df.years_to_end < h)).astype(int)
 
# disjoint bins: 1,2,3 = years before filing, 0 = never-failed firm-year
df["yrs_before_fail"] = np.where(df.failed_co == 1, df.years_to_end + 1, 0)
 
# ------------------------------------------------------- 4. sanity on dist.
print("\n=== 4. Does the last row look distressed? (medians) ===")
ta = df.total_assets.replace(0, np.nan)
tl = df.total_liabilities.replace(0, np.nan)
df["re_ta"] = df.retained_earnings / ta
df["ebit_ta"] = df.ebit / ta
df["wc_ta"] = (df.current_assets - df.current_liabilities) / ta
df["mv_tl"] = df.market_value / tl
cols = ["re_ta", "ebit_ta", "wc_ta", "mv_tl"]
tab = (df[df.failed_co == 1].query("years_to_end <= 5")
       .groupby("years_to_end")[cols].median())
tab.loc["alive (all rows)"] = df[df.failed_co == 0][cols].median()
print(tab.round(3))
tab.to_csv("results/step2_distress_by_years_to_end.csv")
 
# ------------------------------------------------------------- 5. splits
def split_of(y):
    return "train" if y <= 2011 else ("val" if y <= 2014 else "test")
 
 
df["split"] = df["year"].map(split_of)
 
print("\n=== 5. Rows and positives per split ===")
rows = []
for s in ("train", "val", "test"):
    d = df[df.split == s]
    rows.append({"split": s, "rows": len(d),
                 "companies": d.company_name.nunique(),
                 "pos_h1": int(d.y1.sum()), "pos_h2": int(d.y2.sum()),
                 "pos_h3": int(d.y3.sum()),
                 "prev_h1_%": round(100 * d.y1.mean(), 2),
                 "prev_h3_%": round(100 * d.y3.mean(), 2)})
summ = pd.DataFrame(rows)
print(summ.to_string(index=False))
summ.to_csv("results/step2_split_summary.csv", index=False)
 
# companies present in more than one split
nsplit = df.groupby("company_name")["split"].nunique()
print("\nCompanies appearing in >1 split:", int((nsplit > 1).sum()),
      "of", len(nsplit))
print("Failed companies appearing in >1 split:",
      int((nsplit[df.groupby('company_name').failed_co.max() == 1] > 1).sum()))
 
# companies with too little history for lagged features
n_rows = df.groupby("company_name")["year"].transform("size")
print("Rows from companies with a single observation:",
      int((n_rows == 1).sum()))
 
df.to_csv("data/processed/targets.csv", index=False)
print("\nSaved data/processed/targets.csv")
 

=== 1. Is status_label constant within a company? ===
Companies with BOTH labels: 0 of 8971
Companies with a gap in years: 710

=== 2. Accounting-identity checks on the X->name mapping ===
net_sales == total_revenue      : 1.0
ebitda == ebit + D&A            : 0.9999618718385399
gross_profit == net_sales - cogs: 0.9999745812256933
opex == total_revenue - ebitda  : 0.9999618718385399
share CL + LTD <= total_liab    : 0.9999618718385399
share inv + recv <= curr_assets : 0.9998601967413131
share curr_assets <= total_assets: 0.9999872906128466
share total_liab <= 3*total_assets (loose): 0.9725985612973742

=== 4. Does the last row look distressed? (medians) ===
                  re_ta  ebit_ta  wc_ta  mv_tl
years_to_end                                  
0                -0.962   -0.085 -0.016  0.269
1                -0.505   -0.039  0.083  0.575
2                -0.369   -0.010  0.119  0.895
3                -0.238    0.001  0.109  1.046
4                -0.233    0.005  0.125  1.046
5    